In [ ]:
# =========================================================
# PAPER 1 — SUPPLEMENTARY MATERIAL
# STEP 1: Locate input-data/QC files for Fig. S1 and Table S1
# =========================================================

from pathlib import Path

PROJECT = Path(r"C:\project 3\Paper1_SSL")

search_roots = [
    PROJECT / "02_data",
    PROJECT / "03_outputs"
]

keywords = [
    "qc",
    "channel",
    "cube",
    "statistics",
    "summary",
    "correlation",
    "vif",
    "raster",
    "aligned",
    "final"
]

print("Candidate files for Fig. S1 / Table S1:\n")

for root in search_roots:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file():
                name = p.name.lower()

                if any(k in name for k in keywords):
                    print(p)

In [ ]:
# =========================================================
# FIG. S1 — LOCATE THE 14 HARMONIZED INPUT RASTERS
# =========================================================

from pathlib import Path

PROJECT = Path(r"C:\project 3\Paper1_SSL")

RASTER_DIR = PROJECT / "02_data" / "harmonized_4km"

tif_files = sorted(RASTER_DIR.glob("*.tif"))

print("Harmonized 4-km TIFF rasters:\n")

for i, p in enumerate(tif_files, start=1):
    print(f"{i:02d}. {p.name}")

print("\nTotal TIFF files:", len(tif_files))

In [ ]:
# =========================================================
# FIG. S1 — 14-CHANNEL MULTIPHYSICS RASTER ATLAS
# CELL 1: Imports, paths, and authoritative raster order
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

PROJECT = Path(r"C:\project 3\Paper1_SSL")
RASTER_DIR = PROJECT / "02_data" / "harmonized_4km"
FIG_DIR = PROJECT / "03_outputs" / "supplementary_figures"

FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Authoritative ordered raster dictionary
# ---------------------------------------------------------
RASTERS = {
    "RTE_TMI": RASTER_DIR / "TMI_RTE_4km.tif",
    "CBG": RASTER_DIR / "P1_CBG_4km.tif",
    "CBG_RES": RASTER_DIR / "P1_CBG_Res_4km.tif",
    "HGM": RASTER_DIR / "P1_HGM_4km_FINAL.tif",
    "K": RASTER_DIR / "P1_K_4km.tif",
    "eTh": RASTER_DIR / "P1_Th_4km.tif",
    "eU": RASTER_DIR / "P1_U_4km.tif",
    "CPD": RASTER_DIR / "P1_CPD_4km.tif",
    "MAG_LD": RASTER_DIR / "P1_MAG_LD_4km.tif",
    "GRAV_LD": RASTER_DIR / "P1_GRAV_LD_4km.tif",
    "DEM_LD": RASTER_DIR / "P1_DEM_LD_4km.tif",
    "ID": RASTER_DIR / "P1_ID_4km.tif",
    "DEM": RASTER_DIR / "P1_DEM_4km.tif",
    "SLOPE": RASTER_DIR / "P1_SLOPE_4km.tif",
}

# ---------------------------------------------------------
# Display labels
# ---------------------------------------------------------
DISPLAY_LABELS = {
    "RTE_TMI": "RTE-TMI",
    "CBG": "CBG",
    "CBG_RES": "CBG_RES",
    "HGM": "HGM",
    "K": "K",
    "eTh": "eTh",
    "eU": "eU",
    "CPD": "CPD",
    "MAG_LD": "MAG_LD",
    "GRAV_LD": "GRAV_LD",
    "DEM_LD": "DEM_LD",
    "ID": "ID",
    "DEM": "DEM",
    "SLOPE": "SLOPE",
}

# ---------------------------------------------------------
# Colormaps chosen per variable type
# ---------------------------------------------------------
CMAPS = {
    "RTE_TMI": "RdBu_r",
    "CBG": "viridis",
    "CBG_RES": "RdBu_r",
    "HGM": "magma",
    "K": "plasma",
    "eTh": "plasma",
    "eU": "plasma",
    "CPD": "turbo",
    "MAG_LD": "cividis",
    "GRAV_LD": "cividis",
    "DEM_LD": "cividis",
    "ID": "cividis",
    "DEM": "terrain",
    "SLOPE": "YlOrBr",
}

print("Fig. S1 output folder:")
print(FIG_DIR)

print("\nOrdered rasters:")
for k, v in RASTERS.items():
    print(f"{k:8s} -> {v.name}")

In [ ]:
# =========================================================
# FIG. S1 — CELL 2: Quick raster QC summary
# =========================================================

qc_rows = []

for key, path in RASTERS.items():
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        valid = np.isfinite(arr)
        vals = arr[valid]

        qc_rows.append({
            "Channel": key,
            "File": path.name,
            "Rows": src.height,
            "Cols": src.width,
            "CRS": str(src.crs),
            "Xres": src.res[0],
            "Yres": src.res[1],
            "Valid_cells": int(valid.sum()),
            "Min": float(np.nanmin(vals)),
            "Max": float(np.nanmax(vals)),
            "P2": float(np.nanpercentile(vals, 2)),
            "P98": float(np.nanpercentile(vals, 98)),
        })

figs1_qc = pd.DataFrame(qc_rows)
display(figs1_qc)

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

test_path = Path(r"C:\project 3\Paper1_SSL\03_outputs\supplementary_figures\matplotlib_test.png")

fig = Figure(figsize=(2, 2))
FigureCanvasAgg(fig)

ax = fig.add_subplot(111)
ax.plot([0, 1], [0, 1])
ax.set_title("Matplotlib test")

fig.savefig(test_path, dpi=100)
print("SUCCESS:", test_path)

In [ ]:
# =========================================================
# FIG. S1 — 14-CHANNEL MULTIPHYSICS ATLAS
# 4 rasters per row for improved readability
# 4 × 4 layout; final row centered
# =========================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")
RASTER_DIR = PROJECT / "02_data" / "harmonized_4km"

# ---------------------------------------------------------
# Authoritative raster order
# ---------------------------------------------------------
RASTERS = {
    "RTE_TMI": RASTER_DIR / "TMI_RTE_4km.tif",
    "CBG": RASTER_DIR / "P1_CBG_4km.tif",
    "CBG_RES": RASTER_DIR / "P1_CBG_Res_4km.tif",
    "HGM": RASTER_DIR / "P1_HGM_4km_FINAL.tif",

    "K": RASTER_DIR / "P1_K_4km.tif",
    "eTh": RASTER_DIR / "P1_Th_4km.tif",
    "eU": RASTER_DIR / "P1_U_4km.tif",
    "CPD": RASTER_DIR / "P1_CPD_4km.tif",

    "MAG_LD": RASTER_DIR / "P1_MAG_LD_4km.tif",
    "GRAV_LD": RASTER_DIR / "P1_GRAV_LD_4km.tif",
    "DEM_LD": RASTER_DIR / "P1_DEM_LD_4km.tif",
    "ID": RASTER_DIR / "P1_ID_4km.tif",

    "DEM": RASTER_DIR / "P1_DEM_4km.tif",
    "SLOPE": RASTER_DIR / "P1_SLOPE_4km.tif",
}

# ---------------------------------------------------------
# Display labels
# ---------------------------------------------------------
DISPLAY_LABELS = {
    "RTE_TMI": "RTE-TMI",
    "CBG": "CBG",
    "CBG_RES": "CBG_RES",
    "HGM": "HGM",
    "K": "K",
    "eTh": "eTh",
    "eU": "eU",
    "CPD": "CPD",
    "MAG_LD": "MAG_LD",
    "GRAV_LD": "GRAV_LD",
    "DEM_LD": "DEM_LD",
    "ID": "ID",
    "DEM": "DEM",
    "SLOPE": "SLOPE",
}

# ---------------------------------------------------------
# Colormaps
# ---------------------------------------------------------
CMAPS = {
    "RTE_TMI": "RdBu_r",
    "CBG": "viridis",
    "CBG_RES": "RdBu_r",
    "HGM": "magma",
    "K": "plasma",
    "eTh": "plasma",
    "eU": "plasma",
    "CPD": "turbo",
    "MAG_LD": "cividis",
    "GRAV_LD": "cividis",
    "DEM_LD": "cividis",
    "ID": "cividis",
    "DEM": "terrain",
    "SLOPE": "YlOrBr",
}

# ---------------------------------------------------------
# Panel letters
# ---------------------------------------------------------
panel_letters = [
    "(a)", "(b)", "(c)", "(d)",
    "(e)", "(f)", "(g)", "(h)",
    "(i)", "(j)", "(k)", "(l)",
    "(m)", "(n)"
]

# ---------------------------------------------------------
# Create 4 × 4 layout
# ---------------------------------------------------------
fig, axes = plt.subplots(
    nrows=4,
    ncols=4,
    figsize=(15.5, 13.5)
)

axes = axes.ravel()

# ---------------------------------------------------------
# Plot all 14 rasters
# ---------------------------------------------------------
for i, (key, path) in enumerate(RASTERS.items()):

    ax = axes[i]

    with rasterio.open(path) as src:

        arr = src.read(1, masked=True).astype("float64")
        arr = arr.filled(np.nan)

        vals = arr[np.isfinite(arr)]

        # Robust display stretch
        vmin = np.nanpercentile(vals, 2)
        vmax = np.nanpercentile(vals, 98)

        # Symmetric color scale for signed anomaly fields
        if key in ["RTE_TMI", "CBG_RES"]:
            lim = max(abs(vmin), abs(vmax))
            vmin = -lim
            vmax = lim

        im = ax.imshow(
            arr,
            cmap=CMAPS[key],
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
            aspect="auto"
        )

        # Panel letter
        ax.text(
            0.02,
            0.97,
            panel_letters[i],
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            ha="left",
            va="top",
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.82,
                pad=1.2
            )
        )

        # Channel title
        ax.set_title(
            DISPLAY_LABELS[key],
            fontsize=12,
            fontweight="bold",
            pad=5
        )

        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

        # -------------------------------------------------
        # Larger and clearer colorbar legends
        # -------------------------------------------------
        cbar = fig.colorbar(
            im,
            ax=ax,
            fraction=0.050,
            pad=0.025
        )

        cbar.ax.tick_params(
            labelsize=10,
            width=0.9,
            length=3.5
        )

        for tick in cbar.ax.get_yticklabels():
            tick.set_fontweight("bold")

# ---------------------------------------------------------
# Hide unused final two panels
# ---------------------------------------------------------
axes[14].set_visible(False)
axes[15].set_visible(False)

# ---------------------------------------------------------
# Tighten spacing while allowing readable legends
# ---------------------------------------------------------
plt.subplots_adjust(
    left=0.045,
    right=0.985,
    top=0.965,
    bottom=0.045,
    wspace=0.36,
    hspace=0.38
)

plt.show()

In [ ]:
import sys
print(sys.executable)

In [ ]:
# =========================================================
# FIG. S1 — 14-CHANNEL MULTIPHYSICS ATLAS
# Pillow-based 600 dpi PNG with large readable text
# =========================================================

from pathlib import Path
import sys
import numpy as np
import tifffile
from PIL import Image, ImageDraw, ImageFont
from matplotlib import colormaps

print("Active kernel:", sys.executable)

if "paper1_figures" not in sys.executable.lower():
    raise RuntimeError(
        "Select Kernel > Change Kernel > Python (paper1_figures), "
        "then run this cell again."
    )

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")
RASTER_DIR = PROJECT / "02_data" / "harmonized_4km"
OUTPUT_DIR = PROJECT / "03_outputs" / "supplementary_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Raster order, labels, and colormaps
# ---------------------------------------------------------
RASTERS = {
    "RTE_TMI": RASTER_DIR / "TMI_RTE_4km.tif",
    "CBG": RASTER_DIR / "P1_CBG_4km.tif",
    "CBG_RES": RASTER_DIR / "P1_CBG_Res_4km.tif",
    "HGM": RASTER_DIR / "P1_HGM_4km_FINAL.tif",
    "K": RASTER_DIR / "P1_K_4km.tif",
    "eTh": RASTER_DIR / "P1_Th_4km.tif",
    "eU": RASTER_DIR / "P1_U_4km.tif",
    "CPD": RASTER_DIR / "P1_CPD_4km.tif",
    "MAG_LD": RASTER_DIR / "P1_MAG_LD_4km.tif",
    "GRAV_LD": RASTER_DIR / "P1_GRAV_LD_4km.tif",
    "DEM_LD": RASTER_DIR / "P1_DEM_LD_4km.tif",
    "ID": RASTER_DIR / "P1_ID_4km.tif",
    "DEM": RASTER_DIR / "P1_DEM_4km.tif",
    "SLOPE": RASTER_DIR / "P1_SLOPE_4km.tif",
}

DISPLAY_LABELS = {
    "RTE_TMI": "RTE-TMI", "CBG": "CBG", "CBG_RES": "CBG_RES",
    "HGM": "HGM", "K": "K", "eTh": "eTh", "eU": "eU",
    "CPD": "CPD", "MAG_LD": "MAG_LD", "GRAV_LD": "GRAV_LD",
    "DEM_LD": "DEM_LD", "ID": "ID", "DEM": "DEM", "SLOPE": "SLOPE",
}

CMAPS = {
    "RTE_TMI": "RdBu_r", "CBG": "viridis", "CBG_RES": "RdBu_r",
    "HGM": "magma", "K": "plasma", "eTh": "plasma", "eU": "plasma",
    "CPD": "turbo", "MAG_LD": "cividis", "GRAV_LD": "cividis",
    "DEM_LD": "cividis", "ID": "cividis", "DEM": "terrain",
    "SLOPE": "YlOrBr",
}

PANEL_LETTERS = [
    "(a)", "(b)", "(c)", "(d)",
    "(e)", "(f)", "(g)", "(h)",
    "(i)", "(j)", "(k)", "(l)",
    "(m)", "(n)"
]

# ---------------------------------------------------------
# Fonts: pixel sizes are set for a 600 dpi output
# ---------------------------------------------------------
def get_font(size, bold=False):
    font_name = "arialbd.ttf" if bold else "arial.ttf"
    font_path = Path(r"C:\Windows\Fonts") / font_name

    if font_path.exists():
        return ImageFont.truetype(str(font_path), size=size)

    return ImageFont.load_default()

title_font = get_font(120, bold=True)  # about 14 pt at 600 dpi
panel_font = get_font(100, bold=True)  # about 12 pt at 600 dpi
tick_font = get_font(96, bold=True)    # about 11.5 pt at 600 dpi

# ---------------------------------------------------------
# Read GeoTIFF without rasterio
# ---------------------------------------------------------
def read_geotiff(path):
    with tifffile.TiffFile(path) as tif:
        page = tif.pages[0]
        arr = tif.asarray().astype("float64")

        nodata_tag = page.tags.get(42113)

        if nodata_tag is not None:
            try:
                nodata = float(str(nodata_tag.value).replace("\x00", ""))
                arr[arr == nodata] = np.nan
            except (TypeError, ValueError):
                pass

    return arr

# ---------------------------------------------------------
# Colorbar tick formatting
# ---------------------------------------------------------
def format_tick(value):
    av = abs(value)

    if av >= 100:
        return f"{value:.0f}"
    if av >= 10:
        return f"{value:.1f}"
    if av >= 1:
        return f"{value:.1f}"

    return f"{value:.2f}"

# ---------------------------------------------------------
# Final canvas dimensions
# ---------------------------------------------------------
MAP_W = 1200
MAP_H = 730

CBAR_W = 56
LABEL_W = 340

LEFT_MARGIN = 75
TOP_MARGIN = 55
RIGHT_MARGIN = 60
BOTTOM_MARGIN = 60

TITLE_H = 160
GAP_X = 55
GAP_Y = 42

PANEL_W = MAP_W + 18 + CBAR_W + LABEL_W
PANEL_H = TITLE_H + MAP_H

CANVAS_W = (
    LEFT_MARGIN
    + (4 * PANEL_W)
    + (3 * GAP_X)
    + RIGHT_MARGIN
)

CANVAS_H = (
    TOP_MARGIN
    + (4 * PANEL_H)
    + (3 * GAP_Y)
    + BOTTOM_MARGIN
)

canvas = Image.new("RGB", (CANVAS_W, CANVAS_H), "white")
draw = ImageDraw.Draw(canvas)

# ---------------------------------------------------------
# Draw panels
# ---------------------------------------------------------
for i, (key, path) in enumerate(RASTERS.items()):

    row = i // 4
    col = i % 4

    x0 = LEFT_MARGIN + col * (PANEL_W + GAP_X)
    y0 = TOP_MARGIN + row * (PANEL_H + GAP_Y)

    arr = read_geotiff(path)
    vals = arr[np.isfinite(arr)]

    if vals.size == 0:
        raise ValueError(f"No valid values found in {path.name}")

    vmin = np.nanpercentile(vals, 2)
    vmax = np.nanpercentile(vals, 98)

    if key in ["RTE_TMI", "CBG_RES"]:
        limit = max(abs(vmin), abs(vmax))
        vmin, vmax = -limit, limit

    cmap = colormaps[CMAPS[key]]

    # Panel title
    draw.text(
        (x0 + MAP_W // 2, y0 + TITLE_H // 2),
        DISPLAY_LABELS[key],
        font=title_font,
        fill="black",
        anchor="mm"
    )

    # Raster panel
    norm = (arr - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)

    rgba = cmap(norm)
    rgb = (rgba[:, :, :3] * 255).astype(np.uint8)
    rgb[~np.isfinite(arr)] = [255, 255, 255]

    panel_img = Image.fromarray(rgb, mode="RGB").resize(
        (MAP_W, MAP_H),
        resample=Image.Resampling.NEAREST
    )

    map_y = y0 + TITLE_H
    canvas.paste(panel_img, (x0, map_y))

    draw.rectangle(
        [x0, map_y, x0 + MAP_W, map_y + MAP_H],
        outline="black",
        width=4
    )

    # Panel label
    letter_box = [
        x0 + 14,
        map_y + 14,
        x0 + 205,
        map_y + 145
    ]

    draw.rectangle(letter_box, fill="white")

    draw.text(
        (x0 + 110, map_y + 80),
        PANEL_LETTERS[i],
        font=panel_font,
        fill="black",
        anchor="mm"
    )

    # Colorbar
    cbar_x = x0 + MAP_W + 18

    gradient = np.linspace(1, 0, MAP_H)
    cbar_rgba = cmap(gradient)
    cbar_rgb = (cbar_rgba[:, :3] * 255).astype(np.uint8)

    cbar_arr = np.repeat(
        cbar_rgb[:, np.newaxis, :],
        CBAR_W,
        axis=1
    )

    cbar_img = Image.fromarray(cbar_arr, mode="RGB")
    canvas.paste(cbar_img, (cbar_x, map_y))

    draw.rectangle(
        [cbar_x, map_y, cbar_x + CBAR_W, map_y + MAP_H],
        outline="black",
        width=4
    )

    # Large colorbar labels
    tick_values = np.linspace(vmax, vmin, 5)

    for j, value in enumerate(tick_values):
        tick_y = map_y + int(j * MAP_H / 4)

        draw.line(
            [cbar_x + CBAR_W, tick_y, cbar_x + CBAR_W + 15, tick_y],
            fill="black",
            width=3
        )

        draw.text(
            (cbar_x + CBAR_W + 22, tick_y),
            format_tick(value),
            font=tick_font,
            fill="black",
            anchor="lm"
        )

# ---------------------------------------------------------
# Save genuine 600 dpi PNG
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS1_14ch_Multiphysics_Atlas_LargeText_600dpi.png"
)

canvas.save(
    png_path,
    format="PNG",
    dpi=(600, 600),
    optimize=True
)

print("\nSaved successfully:")
print(png_path)

In [ ]:
# =========================================================
# TABLE S1 — LOAD AUTHORITATIVE 14-CHANNEL QC FILES
# =========================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\project 3\Paper1_SSL")

TABLE_DIR = PROJECT / "03_outputs" / "tables"
QC_DIR = PROJECT / "03_outputs" / "qc"

inventory_file = TABLE_DIR / "Paper1_FINAL_14ch_inventory.csv"
stats_file = TABLE_DIR / "Paper1_14ch_descriptive_statistics.csv"
vif_file = TABLE_DIR / "Paper1_14ch_VIF.csv"

geometry_file = QC_DIR / "Paper1_14ch_geometry_QC.csv"
missing_file = QC_DIR / "Paper1_14ch_missing_data_QC.csv"

inventory_df = pd.read_csv(inventory_file)
stats_df = pd.read_csv(stats_file)
vif_df = pd.read_csv(vif_file)

geometry_df = pd.read_csv(geometry_file)
missing_df = pd.read_csv(missing_file)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("INVENTORY")
print("=" * 70)
print("Shape:", inventory_df.shape)
print("Columns:", inventory_df.columns.tolist())
display(inventory_df)

print("\nDESCRIPTIVE STATISTICS")
print("=" * 70)
print("Shape:", stats_df.shape)
print("Columns:", stats_df.columns.tolist())
display(stats_df)

print("\nGEOMETRY QC")
print("=" * 70)
print("Shape:", geometry_df.shape)
print("Columns:", geometry_df.columns.tolist())
display(geometry_df)

print("\nMISSING-DATA QC")
print("=" * 70)
print("Shape:", missing_df.shape)
print("Columns:", missing_df.columns.tolist())
display(missing_df)

print("\nVIF")
print("=" * 70)
print("Shape:", vif_df.shape)
print("Columns:", vif_df.columns.tolist())
display(vif_df)

In [ ]:
# =========================================================
# TABLE S1 — COMPLETE 14-CHANNEL QC AND DESCRIPTIVE STATISTICS
# Final supplementary-table structure
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

TABLE_DIR = PROJECT / "03_outputs" / "tables"
QC_DIR = PROJECT / "03_outputs" / "qc"

inventory_file = TABLE_DIR / "Paper1_FINAL_14ch_inventory.csv"
stats_file = TABLE_DIR / "Paper1_14ch_descriptive_statistics.csv"
missing_file = QC_DIR / "Paper1_14ch_missing_data_QC.csv"
vif_file = TABLE_DIR / "Paper1_14ch_VIF.csv"

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
inventory_df = pd.read_csv(inventory_file)
stats_df = pd.read_csv(stats_file)
missing_df = pd.read_csv(missing_file)
vif_df = pd.read_csv(vif_file)

# ---------------------------------------------------------
# Keep only fields required for Table S1
# ---------------------------------------------------------
inventory_s1 = inventory_df[
    [
        "channel",
        "family",
        "physical_role"
    ]
].copy()

stats_s1 = stats_df[
    [
        "channel",
        "count",
        "min",
        "q1",
        "median",
        "mean",
        "q3",
        "max",
        "std",
        "IQR",
        "skewness",
        "zero_percent"
    ]
].copy()

missing_s1 = missing_df[
    [
        "channel",
        "missing_cells",
        "missing_percent"
    ]
].copy()

vif_s1 = vif_df[
    [
        "channel",
        "VIF",
        "Interpretation"
    ]
].copy()

# ---------------------------------------------------------
# Merge all authoritative outputs
# ---------------------------------------------------------
tableS1 = (
    inventory_s1
    .merge(stats_s1, on="channel", how="left")
    .merge(missing_s1, on="channel", how="left")
    .merge(vif_s1, on="channel", how="left")
)

# ---------------------------------------------------------
# Lock channel order to PB-MMAE input order
# ---------------------------------------------------------
channel_order = [
    "RTE_TMI",
    "CBG",
    "CBG_RES",
    "HGM",
    "K",
    "eTh",
    "eU",
    "CPD",
    "MAG_LD",
    "GRAV_LD",
    "DEM_LD",
    "ID",
    "DEM",
    "SLOPE"
]

tableS1["channel"] = pd.Categorical(
    tableS1["channel"],
    categories=channel_order,
    ordered=True
)

tableS1 = (
    tableS1
    .sort_values("channel")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Rename for manuscript
# ---------------------------------------------------------
tableS1.columns = [
    "Channel",
    "Physics family",
    "Physical role",
    "Common-valid n",
    "Min",
    "Q1",
    "Median",
    "Mean",
    "Q3",
    "Max",
    "SD",
    "IQR",
    "Skewness",
    "Zero (%)",
    "Missing cells",
    "Missing (%)",
    "VIF",
    "VIF interpretation"
]

# ---------------------------------------------------------
# Round numerical values
# ---------------------------------------------------------
round4_cols = [
    "Min",
    "Q1",
    "Median",
    "Mean",
    "Q3",
    "Max",
    "SD",
    "IQR",
    "Skewness",
    "Zero (%)",
    "Missing (%)",
    "VIF"
]

for col in round4_cols:
    tableS1[col] = tableS1[col].round(4)

# Integer formatting
tableS1["Common-valid n"] = tableS1["Common-valid n"].astype(int)
tableS1["Missing cells"] = tableS1["Missing cells"].astype(int)

# ---------------------------------------------------------
# Display full table
# ---------------------------------------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

print("TABLE S1 — 14-CHANNEL QC AND DESCRIPTIVE STATISTICS")
display(tableS1)

print("\nRows:", len(tableS1))
print("Common-valid cells:", tableS1["Common-valid n"].unique())

In [ ]:
# =========================================================
# SAVE TABLE S1
# CSV + Excel
# =========================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\project 3\Paper1_SSL")

SUPP_TABLE_DIR = (
    PROJECT
    / "03_outputs"
    / "tables"
    / "supplementary"
)

SUPP_TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# CSV
# ---------------------------------------------------------
csv_path = (
    SUPP_TABLE_DIR
    / "Paper1_TableS1_14ch_QC_Descriptive_Statistics.csv"
)

tableS1.to_csv(
    csv_path,
    index=False
)

# ---------------------------------------------------------
# Excel
# ---------------------------------------------------------
xlsx_path = (
    SUPP_TABLE_DIR
    / "Paper1_TableS1_14ch_QC_Descriptive_Statistics.xlsx"
)

tableS1.to_excel(
    xlsx_path,
    index=False
)

# ---------------------------------------------------------
# Confirm
# ---------------------------------------------------------
print("Saved Table S1:")
print(csv_path)
print(xlsx_path)

In [ ]:
# =========================================================
# FIG. S2 / TABLE S2 — LOAD AUTHORITATIVE ROBUSTNESS FILES
# =========================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\project 3\Paper1_SSL")

LOCK_DIR = PROJECT / "03_outputs" / "final_results_lock"
PATCH_DIR = PROJECT / "03_outputs" / "patches"
PB_DIR = PROJECT / "03_outputs" / "models" / "PBMMAE" / "Fold3_13x13"

files = {
    "patch_cv": PATCH_DIR / "Paper1_patch_CV_summary.csv",
    "data_cv": LOCK_DIR / "Paper1_FINAL_lock_data_and_spatial_CV.csv",
    "model_decisions": LOCK_DIR / "Paper1_FINAL_lock_model_decisions.csv",
    "sens9": LOCK_DIR / "Paper1_FINAL_lock_9x9_sensitivity.csv",
    "ablations": LOCK_DIR / "Paper1_FINAL_lock_ablations.csv",
    "hierarchy": LOCK_DIR / "Paper1_FINAL_lock_hierarchical_stability.csv",
    "k2to10": PB_DIR / "PBMMAE_Fold3_13x13_consensus_k2to10.csv",
}

dfs = {}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

for name, path in files.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print(path)

    if path.exists():
        df = pd.read_csv(path)
        dfs[name] = df

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df)
    else:
        print("MISSING FILE")

In [ ]:
# =========================================================
# FIG. S2 — SPATIAL FOLD ASSIGNMENT
# Standalone figure: no title and no panel label
# =========================================================

from pathlib import Path
import numpy as np
import rasterio

import matplotlib
matplotlib.use("Agg", force=True)  # Must come before pyplot import

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import FuncFormatter
from pyproj import Transformer
from IPython.display import display, Image

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

INPUT_DIR = PROJECT / "02_data" / "harmonized_4km"
OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS2_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Harmonized rasters used to build the common valid-data mask
# ---------------------------------------------------------
raster_names = [
    "TMI_RTE_4km.tif",
    "P1_CBG_4km.tif",
    "P1_CBG_Res_4km.tif",
    "P1_HGM_4km_FINAL.tif",
    "P1_K_4km.tif",
    "P1_Th_4km.tif",
    "P1_U_4km.tif",
    "P1_CPD_4km.tif",
    "P1_MAG_LD_4km.tif",
    "P1_GRAV_LD_4km.tif",
    "P1_DEM_LD_4km.tif",
    "P1_ID_4km.tif",
    "P1_DEM_4km.tif",
    "P1_SLOPE_4km.tif",
]

raster_paths = [INPUT_DIR / name for name in raster_names]

print("Checking files...")
for path in raster_paths:
    print(path.name, "->", path.exists())

# ---------------------------------------------------------
# Build common valid-data mask across all 14 channels
# ---------------------------------------------------------
common_valid_mask = None

for i, path in enumerate(raster_paths):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        valid = np.isfinite(arr)

        if i == 0:
            ref_shape = arr.shape
            ref_crs = src.crs
            ref_bounds = src.bounds
            common_valid_mask = valid.copy()
        else:
            common_valid_mask &= valid

print("\nReference shape:", ref_shape)
print("CRS:", ref_crs)
print("Bounds:", ref_bounds)
print("Common valid cells:", int(common_valid_mask.sum()))

# ---------------------------------------------------------
# Build five west-east spatial folds
# ---------------------------------------------------------
nrows, ncols = ref_shape
fold_grid = np.full((nrows, ncols), np.nan)

fold_edges = [0, 14, 28, 42, 56, 69]

for fold_id, (c0, c1) in enumerate(
    zip(fold_edges[:-1], fold_edges[1:]), start=1
):
    fold_grid[:, c0:c1] = np.where(
        common_valid_mask[:, c0:c1],
        fold_id,
        np.nan
    )

# ---------------------------------------------------------
# Convert UTM Zone 32N extent to geographic coordinates
# ---------------------------------------------------------
transformer = Transformer.from_crs(ref_crs, "EPSG:4326", always_xy=True)

left, bottom, right, top = ref_bounds
lon_min, lat_min = transformer.transform(left, bottom)
lon_max, lat_max = transformer.transform(right, top)

extent_ll = [lon_min, lon_max, lat_min, lat_max]

print("\nGeographic extent:")
print("Longitude:", lon_min, "to", lon_max)
print("Latitude :", lat_min, "to", lat_max)

# ---------------------------------------------------------
# Degree-minute coordinate formatter
# ---------------------------------------------------------
def degmin_label(value, pos=None, kind="lon"):
    hemi = ("E" if value >= 0 else "W") if kind == "lon" else (
        "N" if value >= 0 else "S"
    )

    value_abs = abs(value)
    degree = int(np.floor(value_abs))
    minutes = int(round((value_abs - degree) * 60))

    if minutes == 60:
        degree += 1
        minutes = 0

    return f"{degree}°{minutes:02d}′{hemi}"


lon_formatter = FuncFormatter(
    lambda x, pos: degmin_label(x, pos, kind="lon")
)
lat_formatter = FuncFormatter(
    lambda y, pos: degmin_label(y, pos, kind="lat")
)

# ---------------------------------------------------------
# Create figure: no panel label
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.6, 6.8))

ax.set_facecolor("#d9d9d9")

cmap = plt.get_cmap("viridis", 5)
bounds = np.arange(0.5, 6.0, 1.0)
norm = BoundaryNorm(bounds, cmap.N)

im = ax.imshow(
    fold_grid,
    cmap=cmap,
    norm=norm,
    extent=extent_ll,
    origin="upper",
    interpolation="nearest",
    aspect="auto"
)

xticks = [8.0, 8.5, 9.0, 9.5, 10.0, 10.5]
yticks = [7 + 29/60, 7 + 52/60, 8 + 15/60, 8 + 37/60, 9.0]

ax.set_xticks(xticks)
ax.set_yticks(yticks)

ax.xaxis.set_major_formatter(lon_formatter)
ax.yaxis.set_major_formatter(lat_formatter)

ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")

ax.tick_params(axis="both", labelsize=13, width=1.2, length=5)

for tick in ax.get_xticklabels():
    tick.set_fontweight("bold")

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

# Spatial-fold legend
cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.045,
    pad=0.03,
    ticks=[1, 2, 3, 4, 5]
)

cbar.set_label("Spatial fold", fontsize=14, fontweight="bold")
cbar.ax.tick_params(labelsize=12, width=1.0, length=4)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

fig.tight_layout()

# ---------------------------------------------------------
# Save only a 600 dpi PNG
# ---------------------------------------------------------
output_png = OUTPUT_DIR / "Paper1_FigS2_Spatial_Fold_Assignment_600dpi.png"

fig.savefig(
    output_png,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig)

print("\nSaved successfully:")
print(output_png)

# Safely display the saved figure in the notebook
display(Image(filename=str(output_png)))

In [ ]:
# =========================================================
# REMOVE THE (b) PANEL LABEL FROM THE EXISTING FIGURE PNG
# =========================================================

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

PROJECT = Path(r"C:\project 3\Paper1_SSL")
OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS2_panels"

# Change this filename only if your existing figure has another name
input_png = OUTPUT_DIR / "Paper1_FigS2a_Spatial_Fold_Assignment_600dpi.png"

# New edited output: keeps the original figure unchanged
output_png = OUTPUT_DIR / "Paper1_FigS2_Spatial_Fold_Assignment_NoPanelLabel_600dpi.png"

if not input_png.exists():
    raise FileNotFoundError(
        f"Figure not found:\n{input_png}\n\n"
        "Check the exact PNG filename in the FigS2_panels folder "
        "and update input_png above."
    )

# Open the existing high-resolution PNG
img = Image.open(input_png).convert("RGB")
width, height = img.size
draw = ImageDraw.Draw(img)

# Sample the original colors from safe locations in the figure
gray_color = img.getpixel((int(0.15 * width), int(0.13 * height)))
purple_color = img.getpixel((int(0.22 * width), int(0.15 * height)))

# Label-box position expressed as fractions of the full image size.
# These values match the current Figure S2 spatial-fold panel.
x0 = int(0.143 * width)
x_split = int(0.166 * width)
x1 = int(0.198 * width)

y0 = int(0.038 * height)
y1 = int(0.088 * height)

# Restore the original background beneath the (b) label:
# gray no-data area on the left and fold-1 purple on the right
draw.rectangle([x0, y0, x_split, y1], fill=gray_color)
draw.rectangle([x_split + 1, y0, x1, y1], fill=purple_color)

# Save as 600 dpi PNG
img.save(output_png, dpi=(600, 600))

print("Saved successfully:")
print(output_png)

# Display the updated figure
display(img)

In [ ]:
# =========================================================
# FIG. S2(b) — PATCH-SIZE FEASIBILITY
# Standalone publication-quality panel
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

PATCH_FILE = (
    PROJECT
    / "03_outputs"
    / "patches"
    / "Paper1_patch_CV_summary.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS2_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
patch_df = pd.read_csv(PATCH_FILE)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.6, 6.8))

x = np.arange(len(patch_df))
width = 0.34

bars_train = ax.bar(
    x - width/2,
    patch_df["mean_TRAIN_patches"],
    width,
    label="Mean training patches"
)

bars_val = ax.bar(
    x + width/2,
    patch_df["mean_VALIDATION_patches"],
    width,
    label="Mean validation patches"
)

# ---------------------------------------------------------
# Values above bars
# ---------------------------------------------------------
for bars in [bars_train, bars_val]:
    for bar in bars:
        h = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width()/2,
            h + 22,
            f"{h:.0f}",
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold"
        )

# ---------------------------------------------------------
# Minimum training support annotation
# Put below each pair, horizontal and readable
# ---------------------------------------------------------
for i, row in patch_df.iterrows():

    ax.text(
        i,
        70,
        f"Min train = {int(row['min_TRAIN_patches'])}",
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold"
    )

# ---------------------------------------------------------
# Highlight retained 13×13
# ---------------------------------------------------------
ax.annotate(
    "Primary scale\n13×13 = 52×52 km",
    xy=(
        1,
        patch_df.loc[
            patch_df["Patch_size"] == 13,
            "mean_TRAIN_patches"
        ].iloc[0]
    ),
    xytext=(0.72, 980),
    fontsize=11,
    fontweight="bold",
    ha="center",
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.2
    )
)

# ---------------------------------------------------------
# Highlight 17×17 exclusion
# ---------------------------------------------------------
ax.annotate(
    "17×17 excluded\nminimum train = 112",
    xy=(
        2,
        patch_df.loc[
            patch_df["Patch_size"] == 17,
            "mean_TRAIN_patches"
        ].iloc[0]
    ),
    xytext=(1.75, 790),
    fontsize=10.5,
    fontweight="bold",
    ha="center",
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.2
    )
)

# ---------------------------------------------------------
# Axes formatting
# ---------------------------------------------------------
ax.set_xticks(x)

ax.set_xticklabels(
    [
        f"{int(p)}×{int(p)}"
        for p in patch_df["Patch_size"]
    ],
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel(
    "Patch size",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of valid patches",
    fontsize=14,
    fontweight="bold"
)

ax.tick_params(
    axis="y",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.set_ylim(
    0,
    1180
)

ax.legend(
    frameon=False,
    fontsize=11,
    loc="upper right"
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.985,
    "(a)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.75,
        pad=1.2
    )
)

# ---------------------------------------------------------
# Frame
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS2b_Patch_Size_Feasibility_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS2b_Patch_Size_Feasibility_600dpi.tif"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    tif_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S2(a) — PATCH-SIZE FEASIBILITY
# Extra-large-text publication panel
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

PATCH_FILE = (
    PROJECT
    / "03_outputs"
    / "patches"
    / "Paper1_patch_CV_summary.csv"
)

OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS2_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
patch_df = pd.read_csv(PATCH_FILE)

patch_df["Patch_size"] = pd.to_numeric(
    patch_df["Patch_size"],
    errors="coerce"
)

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(13.5, 9.8))

x = np.arange(len(patch_df))
width = 0.34

bars_train = ax.bar(
    x - width / 2,
    patch_df["mean_TRAIN_patches"],
    width,
    label="Mean training patches",
    zorder=3
)

bars_val = ax.bar(
    x + width / 2,
    patch_df["mean_VALIDATION_patches"],
    width,
    label="Mean validation patches",
    zorder=3
)

# ---------------------------------------------------------
# Values above bars
# ---------------------------------------------------------
for bars in [bars_train, bars_val]:
    for bar in bars:
        h = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 32,
            f"{h:.0f}",
            ha="center",
            va="bottom",
            fontsize=21,
            fontweight="bold",
            zorder=5
        )

# ---------------------------------------------------------
# Minimum training support
# ---------------------------------------------------------
for i, row in patch_df.iterrows():
    ax.text(
        i,
        78,
        f"Min. train = {int(row['min_TRAIN_patches'])}",
        ha="center",
        va="center",
        fontsize=19,
        fontweight="bold",
        zorder=5
    )

# ---------------------------------------------------------
# Highlight retained 13 × 13 scale
# ---------------------------------------------------------
primary_value = patch_df.loc[
    patch_df["Patch_size"] == 13,
    "mean_TRAIN_patches"
].iloc[0]

ax.annotate(
    "Primary scale\n13 × 13 = 52 × 52 km",
    xy=(1, primary_value),
    xytext=(0.70, 1210),
    fontsize=20,
    fontweight="bold",
    ha="center",
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.8
    ),
    zorder=6
)

# ---------------------------------------------------------
# Highlight 17 × 17 exclusion
# ---------------------------------------------------------
excluded_value = patch_df.loc[
    patch_df["Patch_size"] == 17,
    "mean_TRAIN_patches"
].iloc[0]

ax.annotate(
    "17 × 17 excluded\nminimum train = 112",
    xy=(2, excluded_value),
    xytext=(1.82, 900),
    fontsize=19,
    fontweight="bold",
    ha="center",
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.8
    ),
    zorder=6
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xticks(x)

ax.set_xticklabels(
    [f"{int(p)} × {int(p)}" for p in patch_df["Patch_size"]],
    fontsize=25,
    fontweight="bold"
)

ax.set_xlabel(
    "Patch size",
    fontsize=29,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of valid patches",
    fontsize=29,
    fontweight="bold"
)

ax.tick_params(
    axis="y",
    labelsize=25,
    width=1.5,
    length=7
)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.set_ylim(0, 1400)
ax.margins(x=0.10)

# ---------------------------------------------------------
# Legend and styling
# ---------------------------------------------------------
ax.legend(
    frameon=False,
    fontsize=22,
    loc="upper right"
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

# Panel label
ax.text(
    0.015,
    0.985,
    "(a)",
    transform=ax.transAxes,
    fontsize=34,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.88,
        pad=1.2
    ),
    zorder=7
)

for spine in ax.spines.values():
    spine.set_linewidth(1.3)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS2a_Patch_Size_Feasibility_ExtraLargeText_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS2a_Patch_Size_Feasibility_ExtraLargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S2(b) — FINE-CLUSTER STABILITY ACROSS k = 2–10
# Standalone publication-quality panel
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

K_FILE = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "Fold3_13x13"
    / "PBMMAE_Fold3_13x13_consensus_k2to10.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS2_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
k_df = pd.read_csv(K_FILE)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.6, 6.8))

ax.plot(
    k_df["k"],
    k_df["ARI_mean"],
    marker="o",
    markersize=7,
    linewidth=2.0,
    label="ARI"
)

ax.plot(
    k_df["k"],
    k_df["NMI_mean"],
    marker="s",
    markersize=7,
    linewidth=2.0,
    label="NMI"
)

ax.plot(
    k_df["k"],
    k_df["Mean_within_consensus"],
    marker="^",
    markersize=7,
    linewidth=2.0,
    label="Within consensus"
)

# ---------------------------------------------------------
# Retained k = 9
# ---------------------------------------------------------
ax.axvline(
    9,
    linestyle="--",
    linewidth=1.5
)

ax.text(
    9.10,
    0.965,
    "Retained k = 9",
    fontsize=11,
    fontweight="bold",
    rotation=90,
    va="top"
)

# ---------------------------------------------------------
# Axis formatting
# ---------------------------------------------------------
ax.set_xticks(np.arange(2, 11, 1))

ax.set_xlabel(
    "Number of fine clusters, k",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Consensus metric",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylim(0.35, 1.00)

ax.tick_params(
    axis="both",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_xticklabels():
    tick.set_fontweight("bold")

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
ax.legend(
    frameon=False,
    fontsize=11,
    loc="upper center",
    ncol=3
)

# ---------------------------------------------------------
# Grid
# ---------------------------------------------------------
ax.grid(
    linestyle="--",
    alpha=0.22
)

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.985,
    "(b)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.75,
        pad=1.2
    )
)

# ---------------------------------------------------------
# Frame
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS2c_k2to10_Consensus_Stability_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS2c_k2to10_Consensus_Stability_600dpi.tif"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    tif_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S2(b) — FINE-CLUSTER STABILITY ACROSS k = 2–10
# Extra-large-text publication panel
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

K_FILE = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "Fold3_13x13"
    / "PBMMAE_Fold3_13x13_consensus_k2to10.csv"
)

OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS2_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
k_df = pd.read_csv(K_FILE)

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(13.5, 9.8))

ax.plot(
    k_df["k"],
    k_df["ARI_mean"],
    marker="o",
    markersize=12,
    linewidth=3.0,
    label="ARI",
    zorder=3
)

ax.plot(
    k_df["k"],
    k_df["NMI_mean"],
    marker="s",
    markersize=12,
    linewidth=3.0,
    label="NMI",
    zorder=3
)

ax.plot(
    k_df["k"],
    k_df["Mean_within_consensus"],
    marker="^",
    markersize=12,
    linewidth=3.0,
    label="Within consensus",
    zorder=3
)

# ---------------------------------------------------------
# Retained k = 9
# ---------------------------------------------------------
ax.axvline(
    9,
    linestyle="--",
    linewidth=2.2,
    color="0.30",
    zorder=2
)

ax.text(
    9.14,
    0.968,
    "Retained k = 9",
    fontsize=19,
    fontweight="bold",
    rotation=90,
    va="top",
    ha="left",
    zorder=5
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xticks(np.arange(2, 11, 1))

ax.set_xlabel(
    "Number of fine clusters, k",
    fontsize=29,
    fontweight="bold"
)

ax.set_ylabel(
    "Consensus metric",
    fontsize=29,
    fontweight="bold"
)

ax.set_ylim(0.35, 1.00)
ax.set_xlim(1.7, 10.35)

ax.tick_params(
    axis="both",
    labelsize=25,
    width=1.5,
    length=7
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
ax.legend(
    frameon=False,
    fontsize=20,
    loc="upper center",
    ncol=3,
    handlelength=2.0,
    columnspacing=1.5
)

ax.grid(
    linestyle="--",
    alpha=0.22
)

# Panel label
ax.text(
    0.015,
    0.985,
    "(b)",
    transform=ax.transAxes,
    fontsize=34,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.88,
        pad=1.2
    ),
    zorder=6
)

for spine in ax.spines.values():
    spine.set_linewidth(1.3)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS2b_k2to10_Consensus_Stability_ExtraLargeText_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS2b_k2to10_Consensus_Stability_ExtraLargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S2(c) — PAC AND EMPTY VALIDATION CLUSTERS
# Standalone publication-quality panel
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

K_FILE = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "Fold3_13x13"
    / "PBMMAE_Fold3_13x13_consensus_k2to10.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS2_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
k_df = pd.read_csv(K_FILE)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(8.6, 6.8))

# PAC
line1 = ax1.plot(
    k_df["k"],
    k_df["PAC_0.1_0.9"],
    marker="o",
    markersize=7,
    linewidth=2.0,
    label="PAC"
)

# Secondary axis
ax2 = ax1.twinx()

line2 = ax2.plot(
    k_df["k"],
    k_df["Empty_validation_clusters"],
    marker="s",
    markersize=7,
    linewidth=2.0,
    linestyle="--",
    label="Empty validation clusters"
)

# ---------------------------------------------------------
# Retained k = 9
# ---------------------------------------------------------
ax1.axvline(
    9,
    linestyle="--",
    linewidth=1.5
)

ax1.text(
    9.10,
    0.61,
    "Retained k = 9",
    fontsize=11,
    fontweight="bold",
    rotation=90,
    va="top"
)

# ---------------------------------------------------------
# X axis
# ---------------------------------------------------------
ax1.set_xticks(np.arange(2, 11, 1))

ax1.set_xlabel(
    "Number of fine clusters, k",
    fontsize=14,
    fontweight="bold"
)

# ---------------------------------------------------------
# Left Y axis
# ---------------------------------------------------------
ax1.set_ylabel(
    "PAC (lower is better)",
    fontsize=14,
    fontweight="bold"
)

ax1.set_ylim(0.16, 0.64)

# ---------------------------------------------------------
# Right Y axis
# ---------------------------------------------------------
ax2.set_ylabel(
    "Empty validation clusters",
    fontsize=14,
    fontweight="bold"
)

ax2.set_ylim(0, 5.3)

ax2.set_yticks(
    np.arange(0, 6, 1)
)

# ---------------------------------------------------------
# Tick formatting
# ---------------------------------------------------------
ax1.tick_params(
    axis="both",
    labelsize=12,
    width=1.2,
    length=5
)

ax2.tick_params(
    axis="y",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax1.get_xticklabels():
    tick.set_fontweight("bold")

for tick in ax1.get_yticklabels():
    tick.set_fontweight("bold")

for tick in ax2.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Combined legend
# ---------------------------------------------------------
lines = line1 + line2
labels = [line.get_label() for line in lines]

ax1.legend(
    lines,
    labels,
    frameon=False,
    fontsize=11,
    loc="upper center",
    ncol=2
)

# ---------------------------------------------------------
# Grid
# ---------------------------------------------------------
ax1.grid(
    linestyle="--",
    alpha=0.22
)

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax1.text(
    0.015,
    0.985,
    "(c)",
    transform=ax1.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.75,
        pad=1.2
    )
)

# ---------------------------------------------------------
# Frame
# ---------------------------------------------------------
for spine in ax1.spines.values():
    spine.set_linewidth(1.0)

for spine in ax2.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS2d_PAC_Empty_Validation_Clusters_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS2d_PAC_Empty_Validation_Clusters_600dpi.tif"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    tif_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S2(c) — PAC AND EMPTY VALIDATION CLUSTERS
# Extra-large-text publication panel
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

K_FILE = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "Fold3_13x13"
    / "PBMMAE_Fold3_13x13_consensus_k2to10.csv"
)

OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS2_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
k_df = pd.read_csv(K_FILE)

# =========================================================
# Plot
# =========================================================
fig, ax1 = plt.subplots(figsize=(14.8, 9.8))

# PAC: left axis
line1 = ax1.plot(
    k_df["k"],
    k_df["PAC_0.1_0.9"],
    marker="o",
    markersize=12,
    linewidth=3.0,
    label="PAC",
    zorder=3
)

# Empty validation clusters: right axis
ax2 = ax1.twinx()

line2 = ax2.plot(
    k_df["k"],
    k_df["Empty_validation_clusters"],
    marker="s",
    markersize=12,
    linewidth=3.0,
    linestyle="--",
    label="Empty validation clusters",
    zorder=3
)

# ---------------------------------------------------------
# Retained k = 9
# ---------------------------------------------------------
ax1.axvline(
    9,
    linestyle="--",
    linewidth=2.2,
    color="0.30",
    zorder=2
)

ax1.text(
    9.14,
    0.625,
    "Retained k = 9",
    fontsize=19,
    fontweight="bold",
    rotation=90,
    va="top",
    ha="left",
    zorder=5
)

# ---------------------------------------------------------
# X axis
# ---------------------------------------------------------
ax1.set_xticks(np.arange(2, 11, 1))
ax1.set_xlim(1.7, 10.35)

ax1.set_xlabel(
    "Number of fine clusters, k",
    fontsize=29,
    fontweight="bold"
)

# ---------------------------------------------------------
# Left y axis
# ---------------------------------------------------------
ax1.set_ylabel(
    "PAC (lower is better)",
    fontsize=29,
    fontweight="bold"
)

ax1.set_ylim(0.16, 0.64)

# ---------------------------------------------------------
# Right y axis
# ---------------------------------------------------------
ax2.set_ylabel(
    "Empty validation clusters",
    fontsize=29,
    fontweight="bold"
)

ax2.set_ylim(0, 5.3)
ax2.set_yticks(np.arange(0, 6, 1))

# ---------------------------------------------------------
# Tick formatting
# ---------------------------------------------------------
ax1.tick_params(
    axis="both",
    labelsize=25,
    width=1.5,
    length=7
)

ax2.tick_params(
    axis="y",
    labelsize=25,
    width=1.5,
    length=7
)

for tick in ax1.get_xticklabels() + ax1.get_yticklabels():
    tick.set_fontweight("bold")

for tick in ax2.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Combined legend
# ---------------------------------------------------------
lines = line1 + line2
labels = [line.get_label() for line in lines]

ax1.legend(
    lines,
    labels,
    frameon=False,
    fontsize=20,
    loc="upper center",
    ncol=2,
    handlelength=2.0,
    columnspacing=1.6
)

ax1.grid(
    linestyle="--",
    alpha=0.22
)

# Panel label
ax1.text(
    0.015,
    0.985,
    "(c)",
    transform=ax1.transAxes,
    fontsize=34,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.88,
        pad=1.2
    ),
    zorder=6
)

for spine in ax1.spines.values():
    spine.set_linewidth(1.3)

for spine in ax2.spines.values():
    spine.set_linewidth(1.3)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS2c_PAC_Empty_Validation_Clusters_ExtraLargeText_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS2c_PAC_Empty_Validation_Clusters_ExtraLargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# TABLE S2 — PUBLICATION-FACING VERSION FOR TGRS
#
# S2A  Patch-size feasibility
# S2B  Reduced fine-cluster development diagnostics
#
# This intentionally omits:
# - 9×9 sensitivity table
# - hierarchy stability table
# - ablation table
# because those are already covered in the main manuscript
# and figures.
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

PATCH_DIR = PROJECT / "03_outputs" / "patches"
PB_DIR = PROJECT / "03_outputs" / "models" / "PBMMAE" / "Fold3_13x13"

OUT_DIR = (
    PROJECT
    / "03_outputs"
    / "tables"
    / "supplementary"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Input files
# ---------------------------------------------------------
PATCH_FILE = (
    PATCH_DIR
    / "Paper1_patch_CV_summary.csv"
)

K_FILE = (
    PB_DIR
    / "PBMMAE_Fold3_13x13_consensus_k2to10.csv"
)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
patch_df = pd.read_csv(PATCH_FILE)
k_df = pd.read_csv(K_FILE)

# =========================================================
# TABLE S2A — PATCH-SIZE FEASIBILITY
# =========================================================

S2A = patch_df[
    [
        "Patch_size",
        "min_TRAIN_patches",
        "mean_TRAIN_patches",
        "min_VALIDATION_patches",
        "mean_VALIDATION_patches"
    ]
].copy()

# Physical footprint
S2A["Spatial footprint (km)"] = (
    S2A["Patch_size"].astype(int) * 4
).astype(str) + " × " + (
    S2A["Patch_size"].astype(int) * 4
).astype(str)

# Decision
decision_map = {
    9: "Sensitivity scale",
    13: "Primary retained scale",
    17: "Excluded from primary analysis"
}

S2A["Decision"] = (
    S2A["Patch_size"]
    .astype(int)
    .map(decision_map)
)

# Format patch-size labels
S2A["Patch_size"] = (
    S2A["Patch_size"]
    .astype(int)
    .astype(str)
    + "×"
    + S2A["Patch_size"]
    .astype(int)
    .astype(str)
)

# Rename
S2A = S2A.rename(
    columns={
        "Patch_size": "Patch size",
        "min_TRAIN_patches": "Minimum training patches",
        "mean_TRAIN_patches": "Mean training patches",
        "min_VALIDATION_patches": "Minimum validation patches",
        "mean_VALIDATION_patches": "Mean validation patches"
    }
)

# Reorder
S2A = S2A[
    [
        "Patch size",
        "Spatial footprint (km)",
        "Minimum training patches",
        "Mean training patches",
        "Minimum validation patches",
        "Mean validation patches",
        "Decision"
    ]
]

# Round
S2A["Mean training patches"] = (
    S2A["Mean training patches"].round(1)
)

S2A["Mean validation patches"] = (
    S2A["Mean validation patches"].round(1)
)

# =========================================================
# TABLE S2B — REDUCED k = 2–10 DEVELOPMENT DIAGNOSTICS
# =========================================================

S2B = k_df[
    [
        "k",
        "ARI_mean",
        "NMI_mean",
        "PAC_0.1_0.9",
        "Mean_within_consensus",
        "Silhouette",
        "Davies_Bouldin",
        "Empty_validation_clusters"
    ]
].copy()

S2B = S2B.rename(
    columns={
        "k": "k",
        "ARI_mean": "ARI",
        "NMI_mean": "NMI",
        "PAC_0.1_0.9": "PAC",
        "Mean_within_consensus": "Within consensus",
        "Silhouette": "Silhouette",
        "Davies_Bouldin": "DBI",
        "Empty_validation_clusters": "Empty validation clusters"
    }
)

# Flag retained fine partition
S2B["Decision"] = np.where(
    S2B["k"] == 9,
    "Retained fine partition",
    ""
)

# Round
for col in [
    "ARI",
    "NMI",
    "PAC",
    "Within consensus",
    "Silhouette",
    "DBI"
]:
    S2B[col] = S2B[col].round(4)

S2B["Empty validation clusters"] = (
    S2B["Empty validation clusters"]
    .astype(int)
)

# =========================================================
# DISPLAY
# =========================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

print("=" * 90)
print("TABLE S2A — PATCH-SIZE FEASIBILITY")
display(S2A)

print("\n" + "=" * 90)
print("TABLE S2B — REDUCED FINE-CLUSTER DEVELOPMENT DIAGNOSTICS")
display(S2B)

# =========================================================
# SAVE CSV FILES
# =========================================================

csv_A = (
    OUT_DIR
    / "Paper1_TableS2A_Patch_Size_Feasibility_TGRS.csv"
)

csv_B = (
    OUT_DIR
    / "Paper1_TableS2B_k2to10_Reduced_Diagnostics_TGRS.csv"
)

S2A.to_csv(
    csv_A,
    index=False
)

S2B.to_csv(
    csv_B,
    index=False
)

# =========================================================
# SAVE SINGLE EXCEL WORKBOOK
# =========================================================

xlsx_path = (
    OUT_DIR
    / "Paper1_TableS2_TGRS_Publication_Facing.xlsx"
)

with pd.ExcelWriter(
    xlsx_path,
    engine="openpyxl"
) as writer:

    S2A.to_excel(
        writer,
        sheet_name="S2A_Patch_Feasibility",
        index=False
    )

    S2B.to_excel(
        writer,
        sheet_name="S2B_k_Development",
        index=False
    )

# =========================================================
# CONFIRM
# =========================================================

print("\n" + "=" * 90)
print("PUBLICATION-FACING TABLE S2 SAVED")
print("=" * 90)

print("\nWorkbook:")
print(xlsx_path)

print("\nCSV files:")
print(csv_A)
print(csv_B)

In [ ]:
# =========================================================
# FIG. S3 / TABLE S3 — LOCATE AND INSPECT
# POST HOC EVALUATION OUTPUTS
# =========================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(r"C:\project 3\Paper1_SSL")

# ---------------------------------------------------------
# Search likely output folders for external-validation files
# ---------------------------------------------------------
search_roots = [
    PROJECT / "03_outputs",
    PROJECT / "03_outputs" / "external_validation",
    PROJECT / "03_outputs" / "models" / "PBMMAE",
    PROJECT / "03_outputs" / "tables",
]

keywords = [
    "geology",
    "gpi",
    "spring",
    "warm",
    "external",
    "validation",
    "posthoc",
    "post_hoc",
    "chi",
    "cramer",
    "kruskal",
    "permutation"
]

matches = []

for root in search_roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if not p.is_file():
            continue

        name_lower = p.name.lower()

        if any(k in name_lower for k in keywords):
            matches.append(p)

print("=" * 90)
print("CANDIDATE EXTERNAL-VALIDATION FILES")
print("=" * 90)

for p in sorted(set(matches)):
    print(p.relative_to(PROJECT))

# ---------------------------------------------------------
# Inspect candidate CSV files automatically
# ---------------------------------------------------------
print("\n" + "=" * 90)
print("CSV SCHEMAS")
print("=" * 90)

for p in sorted(set(matches)):
    if p.suffix.lower() != ".csv":
        continue

    print("\n" + "-" * 90)
    print(p.relative_to(PROJECT))

    try:
        df = pd.read_csv(p)

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        display(df.head(20))

    except Exception as e:
        print("Could not read:", e)

In [ ]:
# =========================================================
# FIG. S3(a) — GEOLOGICAL ENRICHMENT HEATMAP (CONSISTENT SCALE)
# Colors and annotations both use log2(enrichment ratio)
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

INPUT_FILE = (
    PROJECT
    / "03_outputs"
    / "external_validation"
    / "geology_domain_validation"
    / "Paper1_geology_domain_enrichment_ratio.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS3_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
df = pd.read_csv(INPUT_FILE)

domain_col = df.columns[0]
df = df.rename(columns={domain_col: "PBMMAE_domain"})
df["PBMMAE_domain"] = df["PBMMAE_domain"].astype(str).str.strip()

domain_order = ["M0", "M1", "M2", "M3"]
df = df.set_index("PBMMAE_domain").reindex(domain_order)

heat_raw = df.apply(pd.to_numeric, errors="coerce")
heat_raw = heat_raw.dropna(axis=1, how="all")

# ---------------------------------------------------------
# Shorter publication labels
# ---------------------------------------------------------
short_labels = {
    "Alluvium": "Alluvium",
    "Asu River Group": "Asu River",
    "Awgu Formation": "Awgu",
    "Bima Formation": "Bima",
    "Eze-Aku Group": "Eze-Aku",
    "Lafia-Wukari Formation": "Lafia-Wukari",
    "Migmatite-Gneiss Complex": "Migmatite-Gneiss",
    "New Basalt": "New Basalt",
    "Nkporo Formation": "Nkporo",
    "Older Basalt": "Older Basalt",
    "Pan-African Older Granite Series": "Older Granite",
    "Peralkaline Younger Granite Series": "Younger Granite",
    "Pindiga Formation": "Pindiga",
    "Yola-Bima-Yolde Formation": "Yola-Bima-Yolde"
}

heat_raw = heat_raw.rename(columns=short_labels)

# ---------------------------------------------------------
# Convert to log2(enrichment ratio)
# Small floor avoids log2(0) problems
# ---------------------------------------------------------
eps = 1e-6
heat_for_log = heat_raw.copy()
heat_for_log = heat_for_log.replace(0, eps)

with np.errstate(divide="ignore", invalid="ignore"):
    heat_log2 = np.log2(heat_for_log)

# clip for display
vmin, vmax = -3, 3
heat_plot = heat_log2.clip(lower=vmin, upper=vmax)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(15.5, 6.4))

im = ax.imshow(
    heat_plot.values,
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax,
    aspect="auto",
    interpolation="nearest"
)

# ticks
ax.set_xticks(np.arange(heat_plot.shape[1]))
ax.set_xticklabels(
    heat_plot.columns,
    rotation=42,
    ha="right",
    fontsize=11,
    fontweight="bold"
)

ax.set_yticks(np.arange(len(domain_order)))
ax.set_yticklabels(
    domain_order,
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel(
    "Lithostratigraphic unit",
    fontsize=14,
    fontweight="bold"
)
ax.set_ylabel(
    "PB-MMAE domain",
    fontsize=14,
    fontweight="bold"
)

# ---------------------------------------------------------
# Annotate with SAME quantity as color = log2 ratio
# ---------------------------------------------------------
for i in range(heat_plot.shape[0]):
    for j in range(heat_plot.shape[1]):
        val = heat_plot.iloc[i, j]

        if pd.notna(val):
            txt_color = "white" if abs(val) > 1.6 else "black"

            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=9,
                fontweight="bold",
                color=txt_color
            )

# ---------------------------------------------------------
# Colorbar
# ---------------------------------------------------------
cbar = fig.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
cbar.set_label(
    r"$\log_{2}$(enrichment ratio)",
    fontsize=13,
    fontweight="bold"
)
cbar.ax.tick_params(labelsize=11)
for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.01, 0.98, "(a)",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=16,
    fontweight="bold",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.2)
)

# style
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.subplots_adjust(left=0.08, right=0.965, top=0.96, bottom=0.28)
plt.show()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = OUTPUT_DIR / "Paper1_FigS3a_Geology_Enrichment_Heatmap_log2_600dpi.png"
tif_path = OUTPUT_DIR / "Paper1_FigS3a_Geology_Enrichment_Heatmap_log2_600dpi.tif"

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S3(a) — GEOLOGICAL ENRICHMENT HEATMAP
# Larger text for manuscript and PowerPoint readability
# Colors and annotations both use log2(enrichment ratio)
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

INPUT_FILE = (
    PROJECT
    / "03_outputs"
    / "external_validation"
    / "geology_domain_validation"
    / "Paper1_geology_domain_enrichment_ratio.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS3_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
df = pd.read_csv(INPUT_FILE)

domain_col = df.columns[0]
df = df.rename(columns={domain_col: "PBMMAE_domain"})
df["PBMMAE_domain"] = df["PBMMAE_domain"].astype(str).str.strip()

domain_order = ["M0", "M1", "M2", "M3"]
df = df.set_index("PBMMAE_domain").reindex(domain_order)

heat_raw = df.apply(pd.to_numeric, errors="coerce")
heat_raw = heat_raw.dropna(axis=1, how="all")

# ---------------------------------------------------------
# Shorter publication labels
# ---------------------------------------------------------
short_labels = {
    "Alluvium": "Alluvium",
    "Asu River Group": "Asu River",
    "Awgu Formation": "Awgu",
    "Bima Formation": "Bima",
    "Eze-Aku Group": "Eze-Aku",
    "Lafia-Wukari Formation": "Lafia-Wukari",
    "Migmatite-Gneiss Complex": "Migmatite-Gneiss",
    "New Basalt": "New Basalt",
    "Nkporo Formation": "Nkporo",
    "Older Basalt": "Older Basalt",
    "Pan-African Older Granite Series": "Older Granite",
    "Peralkaline Younger Granite Series": "Younger Granite",
    "Pindiga Formation": "Pindiga",
    "Yola-Bima-Yolde Formation": "Yola-Bima-Yolde"
}

heat_raw = heat_raw.rename(columns=short_labels)

# ---------------------------------------------------------
# Convert to log2(enrichment ratio)
# ---------------------------------------------------------
eps = 1e-6
heat_for_log = heat_raw.copy().replace(0, eps)

with np.errstate(divide="ignore", invalid="ignore"):
    heat_log2 = np.log2(heat_for_log)

vmin, vmax = -3, 3
heat_plot = heat_log2.clip(lower=vmin, upper=vmax)

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(16.5, 7.2))

im = ax.imshow(
    heat_plot.values,
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax,
    aspect="auto",
    interpolation="nearest"
)

# ---------------------------------------------------------
# Larger ticks and labels
# ---------------------------------------------------------
ax.set_xticks(np.arange(heat_plot.shape[1]))
ax.set_xticklabels(
    heat_plot.columns,
    rotation=38,
    ha="right",
    fontsize=15,
    fontweight="bold"
)

ax.set_yticks(np.arange(len(domain_order)))
ax.set_yticklabels(
    domain_order,
    fontsize=18,
    fontweight="bold"
)

ax.set_xlabel(
    "Lithostratigraphic unit",
    fontsize=20,
    fontweight="bold",
    labelpad=10
)

ax.set_ylabel(
    "PB-MMAE domain",
    fontsize=20,
    fontweight="bold",
    labelpad=10
)

# ---------------------------------------------------------
# Large numerical annotations
# ---------------------------------------------------------
for i in range(heat_plot.shape[0]):
    for j in range(heat_plot.shape[1]):

        val = heat_plot.iloc[i, j]

        if pd.notna(val):
            txt_color = "white" if abs(val) > 1.6 else "black"

            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold",
                color=txt_color
            )

# ---------------------------------------------------------
# Larger colorbar
# ---------------------------------------------------------
cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.020
)

cbar.set_label(
    r"$\log_{2}$(enrichment ratio)",
    fontsize=18,
    fontweight="bold",
    labelpad=10
)

cbar.ax.tick_params(
    labelsize=15,
    width=1.1,
    length=5
)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Larger panel label
# ---------------------------------------------------------
ax.text(
    0.01,
    0.98,
    "(a)",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=23,
    fontweight="bold",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.85,
        pad=1.5
    )
)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.subplots_adjust(
    left=0.090,
    right=0.955,
    top=0.955,
    bottom=0.335
)

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = OUTPUT_DIR / "Paper1_FigS3a_Geology_Enrichment_Heatmap_LargeText_600dpi.png"
tif_path = OUTPUT_DIR / "Paper1_FigS3a_Geology_Enrichment_Heatmap_LargeText_600dpi.tif"

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    tif_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S3(b) — AHP-GPI DISTRIBUTIONS ACROSS PB-MMAE DOMAINS
# Final corrected standalone panel
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

GPI_DIR = (
    PROJECT
    / "03_outputs"
    / "external_validation"
    / "GPI_domain_validation"
)

ANALYSIS_FILE = (
    GPI_DIR
    / "Paper1_GPI_domain_analysis_cells.csv"
)

GLOBAL_FILE = (
    GPI_DIR
    / "Paper1_GPI_domain_global_test_summary.csv"
)

SUMMARY_FILE = (
    GPI_DIR
    / "Paper1_GPI_by_domain_descriptive_statistics.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS3_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load authoritative locked outputs
# ---------------------------------------------------------
gpi_cells = pd.read_csv(ANALYSIS_FILE)
gpi_global = pd.read_csv(GLOBAL_FILE)
gpi_summary = pd.read_csv(SUMMARY_FILE)

print("Loaded:")
print(ANALYSIS_FILE)
print(GLOBAL_FILE)
print(SUMMARY_FILE)

# ---------------------------------------------------------
# Exact authoritative columns
# ---------------------------------------------------------
domain_col = "PBMMAE_domain"
gpi_col = "GPI_continuous"

# ---------------------------------------------------------
# Clean cell-wise data
# ---------------------------------------------------------
plot_df = gpi_cells[
    [domain_col, gpi_col]
].copy()

plot_df[gpi_col] = pd.to_numeric(
    plot_df[gpi_col],
    errors="coerce"
)

plot_df = plot_df.dropna(
    subset=[domain_col, gpi_col]
)

domain_order = ["M0", "M1", "M2", "M3"]

plot_df = plot_df[
    plot_df[domain_col].isin(domain_order)
].copy()

plot_df[domain_col] = pd.Categorical(
    plot_df[domain_col],
    categories=domain_order,
    ordered=True
)

plot_df = plot_df.sort_values(domain_col)

# ---------------------------------------------------------
# Group cell values by domain
# ---------------------------------------------------------
grouped = [
    plot_df.loc[
        plot_df[domain_col] == domain,
        gpi_col
    ].to_numpy()
    for domain in domain_order
]

# ---------------------------------------------------------
# Locked global statistics
# ---------------------------------------------------------
global_row = gpi_global.iloc[0]

H_stat = float(global_row["Kruskal_H"])
df_kw = int(global_row["Kruskal_df"])
eps2 = float(global_row["Epsilon_squared"])
p_perm = float(global_row["Permutation_p"])

# ---------------------------------------------------------
# Locked descriptive statistics
# ---------------------------------------------------------
summary = (
    gpi_summary
    .set_index("PBMMAE_domain")
    .reindex(domain_order)
)

means = summary["Mean"].to_numpy()
medians = summary["Median"].to_numpy()
counts = summary["N"].astype(int).to_numpy()

print("\nDomain summary:")
display(summary)

print("\nGlobal test:")
print(f"H = {H_stat:.6f}")
print(f"df = {df_kw}")
print(f"epsilon² = {eps2:.5f}")
print(f"Permutation p = {p_perm:.6f}")

# =========================================================
# FIG. S3(b) — FINAL CLEAN PLOTTING BLOCK
# =========================================================

fig, ax = plt.subplots(figsize=(8.6, 6.8))

# ---------------------------------------------------------
# Boxplots
# ---------------------------------------------------------
bp = ax.boxplot(
    grouped,
    tick_labels=domain_order,
    widths=0.58,
    patch_artist=False,
    showfliers=False,
    medianprops={"linewidth": 2.0},
    whiskerprops={"linewidth": 1.4},
    capprops={"linewidth": 1.4},
    boxprops={"linewidth": 1.4}
)

# ---------------------------------------------------------
# Mean markers
# ---------------------------------------------------------
x = np.arange(1, len(domain_order) + 1)

ax.scatter(
    x,
    means,
    marker="D",
    s=60,
    zorder=4,
    label="Mean"
)

# Mean labels
for xpos, mean_val in zip(x, means):
    ax.text(
        xpos,
        mean_val + 0.10,
        f"{mean_val:.2f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# ---------------------------------------------------------
# Sample-size labels — fixed baseline
# ---------------------------------------------------------
n_y = 1.24

for xpos, n in zip(x, counts):
    ax.text(
        xpos,
        n_y,
        f"n = {n}",
        ha="center",
        va="bottom",
        fontsize=10.5,
        fontweight="bold"
    )

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xlabel(
    "PB-MMAE domain",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Continuous AHP-GPI",
    fontsize=14,
    fontweight="bold"
)

# Extra lower space for n labels
ax.set_ylim(1.15, 4.78)

ax.tick_params(
    axis="both",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Panel label
ax.text(
    0.015,
    0.985,
    "(b)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.90,
        pad=1.2
    ),
    zorder=10
)

# Statistics box — moved right/down
ax.text(
    0.08,
    0.955,
    stats_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=11.2,
    bbox=dict(
        facecolor="white",
        edgecolor="0.75",
        alpha=0.92,
        pad=4
    ),
    zorder=5
)

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
ax.legend(
    frameon=False,
    fontsize=11,
    loc="upper right"
)

# ---------------------------------------------------------
# Grid and frame
# ---------------------------------------------------------
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS3b_AHP_GPI_by_Domain_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS3b_AHP_GPI_by_Domain_600dpi.tif"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    tif_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

print("\nSaved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S3(b) — CONTINUOUS GPI SCORE ACROSS PB-MMAE DOMAINS
# Large-font publication version
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

GPI_DIR = PROJECT / "03_outputs" / "external_validation" / "GPI_domain_validation"

ANALYSIS_FILE = GPI_DIR / "Paper1_GPI_domain_analysis_cells.csv"
GLOBAL_FILE = GPI_DIR / "Paper1_GPI_domain_global_test_summary.csv"
SUMMARY_FILE = GPI_DIR / "Paper1_GPI_by_domain_descriptive_statistics.csv"

OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS3_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
gpi_cells = pd.read_csv(ANALYSIS_FILE)
gpi_global = pd.read_csv(GLOBAL_FILE)
gpi_summary = pd.read_csv(SUMMARY_FILE)

domain_col = "PBMMAE_domain"
gpi_col = "GPI_continuous"
domain_order = ["M0", "M1", "M2", "M3"]

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------
plot_df = gpi_cells[[domain_col, gpi_col]].copy()
plot_df[gpi_col] = pd.to_numeric(plot_df[gpi_col], errors="coerce")
plot_df = plot_df.dropna(subset=[domain_col, gpi_col])

plot_df = plot_df[plot_df[domain_col].isin(domain_order)].copy()
plot_df[domain_col] = pd.Categorical(
    plot_df[domain_col],
    categories=domain_order,
    ordered=True
)

grouped = [
    plot_df.loc[plot_df[domain_col] == domain, gpi_col].to_numpy()
    for domain in domain_order
]

# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------
global_row = gpi_global.iloc[0]

H_stat = float(global_row["Kruskal_H"])
df_kw = int(global_row["Kruskal_df"])
eta_h2 = float(global_row["Epsilon_squared"])
p_perm = float(global_row["Permutation_p"])

summary = (
    gpi_summary
    .set_index("PBMMAE_domain")
    .reindex(domain_order)
)

means = summary["Mean"].to_numpy()
counts = summary["N"].astype(int).to_numpy()

p_text = "< 0.001" if p_perm < 0.001 else f"= {p_perm:.3f}"

stats_text = (
    f"Kruskal-Wallis H = {H_stat:.2f}\n"
    f"df = {df_kw}\n"
    + r"$\eta_H^2$"
    + f" = {eta_h2:.3f}\n"
    + f"Permutation p {p_text}"
)

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(11.2, 8.5))

bp = ax.boxplot(
    grouped,
    tick_labels=domain_order,
    widths=0.58,
    patch_artist=False,
    showfliers=False,
    medianprops={"linewidth": 2.8},
    whiskerprops={"linewidth": 2.0},
    capprops={"linewidth": 2.0},
    boxprops={"linewidth": 2.0}
)

# Mean markers
x = np.arange(1, len(domain_order) + 1)

ax.scatter(
    x, means,
    marker="D",
    s=115,
    zorder=5,
    label="Mean"
)

# Mean values
for xpos, mean_val in zip(x, means):
    ax.text(
        xpos, mean_val + 0.12,
        f"{mean_val:.2f}",
        ha="center",
        va="bottom",
        fontsize=18,
        fontweight="bold",
        zorder=6
    )

# Sample sizes: deliberately below the lowest M3 whisker
n_y = 1.07

for xpos, n in zip(x, counts):
    ax.text(
        xpos, n_y,
        f"n = {n}",
        ha="center",
        va="bottom",
        fontsize=17,
        fontweight="bold",
        zorder=8,
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.96,
            pad=0.25
        )
    )

# Axes
ax.set_xlabel(
    "PB-MMAE domain",
    fontsize=23,
    fontweight="bold"
)

ax.set_ylabel(
    "Continuous GPI score",
    fontsize=23,
    fontweight="bold"
)

ax.set_ylim(0.94, 4.80)

ax.tick_params(
    axis="both",
    labelsize=20,
    width=1.4,
    length=6
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

# Panel label
ax.text(
    0.015, 0.985, "(b)",
    transform=ax.transAxes,
    fontsize=27,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.90, pad=1.2),
    zorder=10
)

# Statistics box
ax.text(
    0.08, 0.955,
    stats_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=17,
    fontweight="bold",
    bbox=dict(
        facecolor="white",
        edgecolor="0.70",
        alpha=0.94,
        pad=4
    ),
    zorder=7
)

# Legend
ax.legend(
    frameon=False,
    fontsize=18,
    loc="upper right"
)

ax.grid(axis="y", linestyle="--", alpha=0.22)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = OUTPUT_DIR / "Paper1_FigS3b_GPI_by_Domain_LargeText_600dpi.png"
tif_path = OUTPUT_DIR / "Paper1_FigS3b_GPI_by_Domain_LargeText_600dpi.tif"

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S3(c) — WARM-SPRING SPATIAL-TRANSLATION TEST
# Standalone publication-quality panel
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

SPRING_DIR = (
    PROJECT
    / "03_outputs"
    / "external_validation"
    / "warm_spring_domain_validation"
)

DIST_FILE = (
    SPRING_DIR
    / "Paper1_warm_spring_translation_concentration_distribution.csv"
)

SUMMARY_FILE = (
    SPRING_DIR
    / "Paper1_warm_spring_domain_validation_summary.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS3_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load
# ---------------------------------------------------------
dist_df = pd.read_csv(DIST_FILE)
summary_df = pd.read_csv(SUMMARY_FILE)

print("Translation distribution:")
display(dist_df)

print("\nValidation summary:")
display(summary_df)

# ---------------------------------------------------------
# Observed statistics
# ---------------------------------------------------------
row = summary_df.iloc[0]

observed_fraction = float(row["Observed_concentration_fraction"])
p_omnibus = float(row["Spatial_translation_p_omnibus"])
resolved_springs = int(row["Resolved_springs"])
tied_springs = int(row["Tied_springs"])
dominant_domain = str(row["Observed_dominant_domain"])

observed_max_count = int(
    round(observed_fraction * resolved_springs)
)

# ---------------------------------------------------------
# Build the null distribution used by the original test
#
# The distribution CSV contains admissible non-original
# translations. Retain configurations with at least as many
# resolved assignments as in the observed spring configuration.
# This includes both five- and six-resolved-spring translations.
# ---------------------------------------------------------
null_df = dist_df.loc[
    dist_df["N_resolved"] >= resolved_springs
].copy()

if null_df.empty:
    raise ValueError("No admissible translations were found.")

freq = (
    null_df
    .groupby(
        "Concentration_fraction",
        as_index=False
    )["Translation_count"]
    .sum()
    .sort_values("Concentration_fraction")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Verify that the plotted distribution reproduces the test
#
# Adding one includes the original configuration once in
# both the numerator and denominator.
# ---------------------------------------------------------
n_translations = int(freq["Translation_count"].sum())

n_extreme = int(
    freq.loc[
        freq["Concentration_fraction"] >= observed_fraction,
        "Translation_count"
    ].sum()
)

p_check = (n_extreme + 1) / (n_translations + 1)

print("\nAdmissible non-original translations:", n_translations)
print("Translations at least as concentrated as observed:", n_extreme)
print("Observed concentration:", observed_fraction)
print("Resolved springs:", resolved_springs)
print("Tied springs:", tied_springs)
print("Recomputed spatial-translation p:", p_check)
print("Stored spatial-translation p:", p_omnibus)

if not np.isclose(p_check, p_omnibus, rtol=0, atol=1e-6):
    raise ValueError(
        "The plotted distribution does not reproduce the stored "
        "spatial-translation probability. Check that the distribution "
        "CSV contains all admissible non-original translations and "
        "corresponds to the supplied summary file."
    )

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.6, 6.8))

ax.bar(
    freq["Concentration_fraction"],
    freq["Translation_count"],
    width=0.025,
    color="#4477AA",
    edgecolor="white",
    linewidth=0.6,
    zorder=3
)

ax.axvline(
    observed_fraction,
    color="#AA3333",
    linestyle="--",
    linewidth=1.8,
    zorder=4
)

# Reserve space above the bars for annotations.
ymax = float(freq["Translation_count"].max())
ax.set_ylim(0, ymax * 1.55)

# ---------------------------------------------------------
# Observed-statistic annotation
# ---------------------------------------------------------
ax.text(
    0.98,
    0.96,
    (
        f"Observed: {observed_max_count}/{resolved_springs} "
        f"resolved springs in {dominant_domain}\n"
        f"Concentration = {observed_fraction:.2f}\n"
        f"Spatial translation p = {p_check:.3f}"
    ),
    transform=ax.transAxes,
    fontsize=11.2,
    fontweight="bold",
    ha="right",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="0.75",
        alpha=0.95,
        pad=5
    ),
    zorder=5
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xlabel(
    "Maximum same-domain concentration fraction",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of spatial translations",
    fontsize=14,
    fontweight="bold"
)

tick_positions = freq["Concentration_fraction"].to_numpy()

ax.set_xticks(tick_positions)
ax.set_xticklabels(
    [f"{x:.2f}" for x in tick_positions],
    fontsize=11,
    fontweight="bold",
    rotation=45,
    ha="right"
)

ax.set_xlim(
    min(tick_positions.min(), observed_fraction) - 0.04,
    max(tick_positions.max(), observed_fraction) + 0.04
)

ax.tick_params(
    axis="y",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.985,
    "(c)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.85,
        pad=1.2
    )
)

# ---------------------------------------------------------
# Interpretation note
# ---------------------------------------------------------
interpretation = (
    "Not significant at α = 0.05"
    if p_check >= 0.05
    else "Significant at α = 0.05"
)

ax.text(
    0.98,
    0.76,
    interpretation,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=11,
    fontweight="bold"
)

# ---------------------------------------------------------
# Grid/frame
# ---------------------------------------------------------
ax.set_axisbelow(True)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

fig.tight_layout()

# ---------------------------------------------------------
# Save before displaying
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS3c_WarmSpring_SpatialTranslation_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS3c_WarmSpring_SpatialTranslation_600dpi.tif"
)

for output_path in (png_path, tif_path):
    fig.savefig(
        output_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()

print("\nSaved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# FIG. S3(c) — WARM-SPRING SPATIAL-TRANSLATION TEST
# Standalone preview with statistical consistency check
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

SPRING_DIR = (
    PROJECT
    / "03_outputs"
    / "external_validation"
    / "warm_spring_domain_validation"
)

DIST_FILE = (
    SPRING_DIR
    / "Paper1_warm_spring_translation_concentration_distribution.csv"
)

SUMMARY_FILE = (
    SPRING_DIR
    / "Paper1_warm_spring_domain_validation_summary.csv"
)

OUTPUT_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "FigS3_panels"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load saved results
# ---------------------------------------------------------
print("Loading saved translation results...", flush=True)

dist_df = pd.read_csv(DIST_FILE)
summary_df = pd.read_csv(SUMMARY_FILE)

row = summary_df.iloc[0]

observed_fraction = float(row["Observed_concentration_fraction"])
p_omnibus = float(row["Spatial_translation_p_omnibus"])
resolved_springs = int(row["Resolved_springs"])
tied_springs = int(row["Tied_springs"])
dominant_domain = str(row["Observed_dominant_domain"])

observed_max_count = int(
    round(observed_fraction * resolved_springs)
)

# ---------------------------------------------------------
# Reconstruct the distribution used by the original test
#
# The CSV contains admissible non-original translations.
# Include translations with at least as many resolved
# springs as in the observed configuration.
# ---------------------------------------------------------
null_df = dist_df.loc[
    dist_df["N_resolved"] >= resolved_springs
].copy()

if null_df.empty:
    raise ValueError("No admissible translations were found.")

freq = (
    null_df
    .groupby(
        "Concentration_fraction",
        as_index=False
    )["Translation_count"]
    .sum()
    .sort_values("Concentration_fraction")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Verify the empirical probability
# The added unit includes the original configuration.
# ---------------------------------------------------------
n_translations = int(freq["Translation_count"].sum())

n_extreme = int(
    freq.loc[
        freq["Concentration_fraction"] >= observed_fraction,
        "Translation_count"
    ].sum()
)

p_check = (n_extreme + 1) / (n_translations + 1)

print(
    "Admissible non-original translations:",
    n_translations,
    flush=True
)
print(
    "Translations at least as concentrated as observed:",
    n_extreme,
    flush=True
)
print("Observed concentration:", observed_fraction, flush=True)
print("Resolved springs:", resolved_springs, flush=True)
print("Tied springs:", tied_springs, flush=True)
print("Recomputed spatial-translation p:", p_check, flush=True)
print("Stored spatial-translation p:", p_omnibus, flush=True)

if not np.isclose(p_check, p_omnibus, rtol=0, atol=1e-6):
    raise ValueError(
        "The distribution does not reproduce the stored probability. "
        "Check that the distribution CSV contains all admissible "
        "non-original translations and matches the summary file."
    )

# ---------------------------------------------------------
# Create the figure using a noninteractive renderer
# ---------------------------------------------------------
print("Creating preview figure...", flush=True)

fig = Figure(figsize=(8.6, 6.8), dpi=150)
canvas = FigureCanvasAgg(fig)
ax = fig.add_subplot(111)

ax.bar(
    freq["Concentration_fraction"],
    freq["Translation_count"],
    width=0.025,
    color="#4477AA",
    edgecolor="white",
    linewidth=0.6,
    zorder=3
)

ax.axvline(
    observed_fraction,
    color="#AA3333",
    linestyle="--",
    linewidth=1.8,
    zorder=4
)

ymax = float(freq["Translation_count"].max())
ax.set_ylim(0, ymax * 1.55)

# ---------------------------------------------------------
# Observed-statistic annotation
# ---------------------------------------------------------
ax.text(
    0.98,
    0.96,
    (
        f"Observed: {observed_max_count}/{resolved_springs} "
        f"resolved springs in {dominant_domain}\n"
        f"Concentration = {observed_fraction:.2f}\n"
        f"Spatial translation p = {p_check:.3f}"
    ),
    transform=ax.transAxes,
    fontsize=11.2,
    fontweight="bold",
    ha="right",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="0.75",
        alpha=0.95,
        pad=5
    ),
    zorder=5
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xlabel(
    "Maximum same-domain concentration fraction",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of spatial translations",
    fontsize=14,
    fontweight="bold"
)

tick_positions = freq["Concentration_fraction"].to_numpy()

ax.set_xticks(tick_positions)
ax.set_xticklabels(
    [f"{x:.2f}" for x in tick_positions],
    fontsize=11,
    fontweight="bold",
    rotation=45,
    ha="right"
)

ax.set_xlim(
    min(tick_positions.min(), observed_fraction) - 0.04,
    max(tick_positions.max(), observed_fraction) + 0.04
)

ax.tick_params(
    axis="y",
    labelsize=12,
    width=1.2,
    length=5
)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Panel label and interpretation
# ---------------------------------------------------------
ax.text(
    0.015,
    0.985,
    "(c)",
    transform=ax.transAxes,
    fontsize=16,
    fontweight="bold",
    ha="left",
    va="top"
)

interpretation = (
    "Not significant at α = 0.05"
    if p_check >= 0.05
    else "Significant at α = 0.05"
)

ax.text(
    0.98,
    0.76,
    interpretation,
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold",
    ha="right",
    va="top"
)

# ---------------------------------------------------------
# Grid and layout
# ---------------------------------------------------------
ax.set_axisbelow(True)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

fig.tight_layout()

# ---------------------------------------------------------
# Save preview only; no inline display or TIFF export
# ---------------------------------------------------------
preview_path = (
    OUTPUT_DIR
    / "Paper1_FigS3c_WarmSpring_SpatialTranslation_preview.png"
)

print("Saving 150-dpi preview...", flush=True)

fig.savefig(
    preview_path,
    dpi=150,
    facecolor="white"
)

fig.clear()
del ax, canvas, fig

print("\nPreview saved successfully:", flush=True)
print(preview_path, flush=True)

In [ ]:
# =========================================================
# FIG. S3(c) — WARM-SPRING SPATIAL-TRANSLATION TEST
# Large-text publication version
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

SPRING_DIR = (
    PROJECT
    / "03_outputs"
    / "external_validation"
    / "warm_spring_domain_validation"
)

DIST_FILE = (
    SPRING_DIR
    / "Paper1_warm_spring_translation_concentration_distribution.csv"
)

SUMMARY_FILE = (
    SPRING_DIR
    / "Paper1_warm_spring_domain_validation_summary.csv"
)

OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "FigS3_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load results
# ---------------------------------------------------------
dist_df = pd.read_csv(DIST_FILE)
summary_df = pd.read_csv(SUMMARY_FILE)

row = summary_df.iloc[0]

observed_fraction = float(row["Observed_concentration_fraction"])
p_omnibus = float(row["Spatial_translation_p_omnibus"])
resolved_springs = int(row["Resolved_springs"])
tied_springs = int(row["Tied_springs"])
dominant_domain = str(row["Observed_dominant_domain"])

observed_max_count = int(round(observed_fraction * resolved_springs))

# ---------------------------------------------------------
# Reconstruct admissible translation distribution
# ---------------------------------------------------------
null_df = dist_df.loc[
    dist_df["N_resolved"] >= resolved_springs
].copy()

if null_df.empty:
    raise ValueError("No admissible translations were found.")

freq = (
    null_df
    .groupby("Concentration_fraction", as_index=False)["Translation_count"]
    .sum()
    .sort_values("Concentration_fraction")
    .reset_index(drop=True)
)

# Verify stored probability
n_translations = int(freq["Translation_count"].sum())

n_extreme = int(
    freq.loc[
        freq["Concentration_fraction"] >= observed_fraction,
        "Translation_count"
    ].sum()
)

p_check = (n_extreme + 1) / (n_translations + 1)

if not np.isclose(p_check, p_omnibus, rtol=0, atol=1e-6):
    raise ValueError(
        "The plotted distribution does not reproduce the stored "
        "spatial-translation probability."
    )

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(13.0, 8.8))

ax.bar(
    freq["Concentration_fraction"],
    freq["Translation_count"],
    width=0.025,
    color="#4477AA",
    edgecolor="white",
    linewidth=0.9,
    zorder=3
)

ax.axvline(
    observed_fraction,
    color="#AA3333",
    linestyle="--",
    linewidth=2.5,
    zorder=4
)

ymax = float(freq["Translation_count"].max())
ax.set_ylim(0, ymax * 1.55)

# ---------------------------------------------------------
# Results annotation
# ---------------------------------------------------------
ax.text(
    0.98,
    0.96,
    (
        f"Observed: {observed_max_count}/{resolved_springs} "
        f"resolved springs in {dominant_domain}\n"
        f"Concentration = {observed_fraction:.2f}\n"
        f"Spatial translation p = {p_check:.3f}"
    ),
    transform=ax.transAxes,
    fontsize=18,
    fontweight="bold",
    ha="right",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="0.70",
        alpha=0.96,
        pad=5
    ),
    zorder=5
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xlabel(
    "Maximum same-domain concentration fraction",
    fontsize=24,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of spatial translations",
    fontsize=24,
    fontweight="bold"
)

tick_positions = freq["Concentration_fraction"].to_numpy()

ax.set_xticks(tick_positions)
ax.set_xticklabels(
    [f"{x:.2f}" for x in tick_positions],
    fontsize=18,
    fontweight="bold",
    rotation=45,
    ha="right"
)

ax.set_xlim(
    min(tick_positions.min(), observed_fraction) - 0.04,
    max(tick_positions.max(), observed_fraction) + 0.04
)

ax.tick_params(
    axis="y",
    labelsize=20,
    width=1.4,
    length=6
)

for tick in ax.get_yticklabels():
    tick.set_fontweight("bold")

# Panel label
ax.text(
    0.015,
    0.985,
    "(c)",
    transform=ax.transAxes,
    fontsize=28,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.90,
        pad=1.2
    ),
    zorder=6
)

# ---------------------------------------------------------
# Styling
# ---------------------------------------------------------
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", alpha=0.22)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

fig.tight_layout()

# ---------------------------------------------------------
# Save final figure
# ---------------------------------------------------------
png_path = (
    OUTPUT_DIR
    / "Paper1_FigS3c_WarmSpring_SpatialTranslation_LargeText_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_FigS3c_WarmSpring_SpatialTranslation_LargeText_600dpi.tif"
)

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# ============================================================
# TRACE PB-MMAE HIERARCHY / META-DOMAIN IMPLEMENTATION
# Searches all .ipynb and .py files in the Paper1_SSL project
# ============================================================

from pathlib import Path
import json
import re

PROJECT = Path(r"C:\project 3\Paper1_SSL")

SEARCH_TERMS = [
    "AgglomerativeClustering",
    "linkage",
    "ward",
    "fcluster",
    "dendrogram",
    "scipy.cluster",
    "hierarchy",
    "hierarchical",
    "meta_domain",
    "meta-domain",
    "metadomain",
    "n_meta",
    "candidate_meta",
    "fine_to_meta",
    "cluster_to_meta",
    "fine_cluster",
    "consensus_centroid",
    "centroids",
    "pdist",
    "squareform",
]

pattern = re.compile(
    "|".join(re.escape(term) for term in SEARCH_TERMS),
    flags=re.IGNORECASE
)

matches = []

# ------------------------------------------------------------
# Search Python scripts
# ------------------------------------------------------------
for path in PROJECT.rglob("*.py"):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    lines = text.splitlines()

    for i, line in enumerate(lines):
        if pattern.search(line):
            start = max(0, i - 8)
            end = min(len(lines), i + 12)

            matches.append({
                "file": str(path.relative_to(PROJECT)),
                "type": "PY",
                "location": f"line {i+1}",
                "snippet": "\n".join(
                    f"{j+1:04d}: {lines[j]}"
                    for j in range(start, end)
                )
            })


# ------------------------------------------------------------
# Search Jupyter notebooks, including code + markdown cells
# ------------------------------------------------------------
for path in PROJECT.rglob("*.ipynb"):
    try:
        nb = json.loads(
            path.read_text(encoding="utf-8", errors="ignore")
        )
    except Exception:
        continue

    for cell_index, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))

        if pattern.search(source):
            matches.append({
                "file": str(path.relative_to(PROJECT)),
                "type": cell.get("cell_type", "unknown").upper(),
                "location": f"cell {cell_index}",
                "snippet": source
            })


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
print("=" * 100)
print("PB-MMAE HIERARCHY / META-DOMAIN CODE TRACE")
print("=" * 100)

print(f"\nProject searched : {PROJECT}")
print(f"Matches found    : {len(matches)}")

for n, m in enumerate(matches, start=1):
    print("\n" + "#" * 100)
    print(f"MATCH {n}")
    print("#" * 100)
    print(f"FILE     : {m['file']}")
    print(f"TYPE     : {m['type']}")
    print(f"LOCATION : {m['location']}")
    print("-" * 100)
    print(m["snippet"])

print("\n" + "=" * 100)
print("TRACE COMPLETE")
print("=" * 100)